# Download & Cache VLMs to HuggingFace Hub Cache

Downloads (snapshots) all models listed in the paper to the default HF cache
(`~/.cache/huggingface/hub`), so they are available offline for inference.

- Toggle any model group on/off in cell 2 before running cell 3.
- Each model is downloaded once; re-running skips already-cached files.
- `HF_TOKEN` is read from the environment — set it below if needed (required for
  gated models such as Llama-4 and Gemma-4).


In [1]:
import os
# Set your HuggingFace token here if not already set in the environment.
# Required for gated models (Llama-4, Gemma-4).
HF_TOKEN = os.environ.get("HF_TOKEN", "")

# ── Model groups — toggle True/False to include/exclude a group ──────────────
DOWNLOAD_GROUPS = {
    "Qwen3-VL":       True,
    "Llama-4":        True,
    "InternVL3_5":    True,
    "DeepSeek-VL2":   True,
    "GLM-4.6V":       True,
    "Gemma-4":        True,
}

# ── Model registry ────────────────────────────────────────────────────────────
# (display_name, hf_model_id, group)
MODEL_REGISTRY = [
    # Qwen3-VL
    ("Qwen3-VL-2B-Instruct",          "Qwen/Qwen3-VL-2B-Instruct",                       "Qwen3-VL"),
    ("Qwen3-VL-4B-Instruct",          "Qwen/Qwen3-VL-4B-Instruct",                       "Qwen3-VL"),
    ("Qwen3-VL-8B-Instruct",          "Qwen/Qwen3-VL-8B-Instruct",                       "Qwen3-VL"),
    ("Qwen3-VL-30B-A3B-Instruct",     "Qwen/Qwen3-VL-30B-A3B-Instruct",                  "Qwen3-VL"),
    # Llama-4
    ("Llama-4-Scout-17B-16E",         "meta-llama/Llama-4-Scout-17B-16E-Instruct",        "Llama-4"),
    ("Llama-4-Maverick-17B-128E",     "meta-llama/Llama-4-Maverick-17B-128E-Instruct",    "Llama-4"),
    # InternVL3.5-Flash
    ("InternVL3_5-1B-Flash",          "OpenGVLab/InternVL3_5-1B-Flash",                  "InternVL3_5"),
    ("InternVL3_5-2B-Flash",          "OpenGVLab/InternVL3_5-2B-Flash",                  "InternVL3_5"),
    ("InternVL3_5-4B-Flash",          "OpenGVLab/InternVL3_5-4B-Flash",                  "InternVL3_5"),
    ("InternVL3_5-8B-Flash",          "OpenGVLab/InternVL3_5-8B-Flash",                  "InternVL3_5"),
    ("InternVL3_5-14B-Flash",         "OpenGVLab/InternVL3_5-14B-Flash",                 "InternVL3_5"),
    ("InternVL3_5-30B-A3B-Flash",     "OpenGVLab/InternVL3_5-30B-A3B-Flash",             "InternVL3_5"),
    ("InternVL3_5-38B-Flash",         "OpenGVLab/InternVL3_5-38B-Flash",                 "InternVL3_5"),
    # DeepSeek-VL2
    ("deepseek-vl2-tiny",             "deepseek-ai/deepseek-vl2-tiny",                   "DeepSeek-VL2"),
    ("deepseek-vl2-small",            "deepseek-ai/deepseek-vl2-small",                  "DeepSeek-VL2"),
    # GLM-4.6V
    ("GLM-4.6V-Flash",                "zai-org/GLM-4.6V-Flash",                          "GLM-4.6V"),
    ("GLM-4.6V",                      "zai-org/GLM-4.6V",                                "GLM-4.6V"),
    # Gemma-4
    ("gemma-4-E2B-it",                "google/gemma-4-E2B-it",                           "Gemma-4"),
    ("gemma-4-E4B-it",                "google/gemma-4-E4B-it",                           "Gemma-4"),
    ("gemma-4-26B-A4B-it",            "google/gemma-4-26B-A4B-it",                       "Gemma-4"),
    ("gemma-4-31B-it",                "google/gemma-4-31B-it",                           "Gemma-4"),
]

selected = [(name, mid) for name, mid, grp in MODEL_REGISTRY if DOWNLOAD_GROUPS.get(grp, False)]
print(f"Will download {len(selected)} models:")
for name, mid in selected:
    print(f"  {name:40s}  {mid}")


Will download 21 models:
  Qwen3-VL-2B-Instruct                      Qwen/Qwen3-VL-2B-Instruct
  Qwen3-VL-4B-Instruct                      Qwen/Qwen3-VL-4B-Instruct
  Qwen3-VL-8B-Instruct                      Qwen/Qwen3-VL-8B-Instruct
  Qwen3-VL-30B-A3B-Instruct                 Qwen/Qwen3-VL-30B-A3B-Instruct
  Llama-4-Scout-17B-16E                     meta-llama/Llama-4-Scout-17B-16E-Instruct
  Llama-4-Maverick-17B-128E                 meta-llama/Llama-4-Maverick-17B-128E-Instruct
  InternVL3_5-1B-Flash                      OpenGVLab/InternVL3_5-1B-Flash
  InternVL3_5-2B-Flash                      OpenGVLab/InternVL3_5-2B-Flash
  InternVL3_5-4B-Flash                      OpenGVLab/InternVL3_5-4B-Flash
  InternVL3_5-8B-Flash                      OpenGVLab/InternVL3_5-8B-Flash
  InternVL3_5-14B-Flash                     OpenGVLab/InternVL3_5-14B-Flash
  InternVL3_5-30B-A3B-Flash                 OpenGVLab/InternVL3_5-30B-A3B-Flash
  InternVL3_5-38B-Flash                     OpenGVLab/Inte

In [ ]:
import gc
import time
import torch
from transformers import AutoProcessor, AutoTokenizer, AutoModel
from huggingface_hub.utils import RepositoryNotFoundError, GatedRepoError

results = []

for name, model_id in selected:
    print(f"\n{'─'*60}")
    print(f"⬇  {name}  ({model_id})")
    t0 = time.time()
    try:
        # Load processor/tokenizer first (fast, downloads config + tokenizer files).
        try:
            AutoProcessor.from_pretrained(model_id, token=HF_TOKEN or None, trust_remote_code=True)
        except Exception:
            AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN or None, trust_remote_code=True)

        # Load model weights onto CPU only — enough to populate the HF cache.
        model = AutoModel.from_pretrained(
            model_id,
            token=HF_TOKEN or None,
            trust_remote_code=True,
            device_map="cpu",          # CPU only; we just want the files cached
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True,
        )
        elapsed = time.time() - t0
        print(f"   ✓  cached in {elapsed:.1f}s")
        results.append({"model": name, "status": "ok", "elapsed_s": round(elapsed, 1)})

        # Free memory immediately.
        del model
        gc.collect()

    except GatedRepoError:
        print(f"   ✗  GATED — accept the licence at https://huggingface.co/{model_id}")
        results.append({"model": name, "status": "gated", "elapsed_s": None})
    except RepositoryNotFoundError:
        print(f"   ✗  NOT FOUND — check the model ID")
        results.append({"model": name, "status": "not_found", "elapsed_s": None})
    except Exception as exc:
        print(f"   ✗  ERROR: {exc}")
        results.append({"model": name, "status": f"error: {exc}", "elapsed_s": None})

print(f"\n{'═'*60}")
print(f"Done.  {sum(r['status']=='ok' for r in results)}/{len(results)} succeeded.\n")
for r in results:
    icon = "✓" if r["status"] == "ok" else "✗"
    print(f"  {icon}  {r['model']:40s}  {r['status']}")



────────────────────────────────────────────────────────────
⬇  Qwen3-VL-2B-Instruct  (Qwen/Qwen3-VL-2B-Instruct)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

   ✓  cached in 5.0s

────────────────────────────────────────────────────────────
⬇  Qwen3-VL-4B-Instruct  (Qwen/Qwen3-VL-4B-Instruct)


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/881 [00:00<?, ?it/s]